In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import scarf
from scarf.agent import (
    AgentRunConfig,
    BiologicalContext,
    BiologicalInterpretationAgent,
    DataEnrichmentAgent,
    DataEnrichmentContext,
    ExperimentalContextAgent,
    ParameterCandidate,
    ParameterTuningAgent,
)

scarf.configure_output(level="WARNING", progress=True)

dataset = scarf.cytebase.connect("scarf_docs").download_dataset(
    "tenx_5K_pbmc_rnaseq",
    destination="scarf_datasets",
    zarr=True,
)
analysis_directory = TemporaryDirectory()
ds = scarf.mount_datastore(
    f"{dataset}/data.zarr",
    at=str(Path(analysis_directory.name) / "agent_analysis.zarr"),
    default_assay="RNA",
    nthreads=2,
    min_features_per_cell=10,
)

{
    "active_cells": int(ds.cells.fetch_all("I").sum()),
    "total_cells": ds.cells.N,
    "assays": ds.assay_names,
}

Downloading bucket files: 48357614 / 48357614 complete

Downloading bytes: 48357614 / 48357614 complete

{'active_cells': 3940, 'total_cells': 5025, 'assays': ['RNA']}

In [2]:
from pydantic_ai.messages import (
    ModelMessage,
    ModelResponse,
    ToolCallPart,
    ToolReturnPart,
)
from pydantic_ai.models.function import AgentInfo, FunctionModel

from scarf.agent.biological_interpretation import (
    ClusterCompositionEvidence,
    ClusterMarkerBatchEvidence,
)
from scarf.agent.data_enrichment import AssayFeatureInspectionBatch
from scarf.agent.experimental_context import CovariateEvidence


def _tool_returns(messages: list[ModelMessage]) -> list[ToolReturnPart]:
    return [
        part
        for message in messages
        for part in message.parts
        if isinstance(part, ToolReturnPart)
    ]


def _structured_output(info: AgentInfo, payload: dict) -> ModelResponse:
    return ModelResponse(
        parts=[
            ToolCallPart(
                tool_name=info.output_tools[0].name,
                args=payload,
            )
        ]
    )


async def _enrichment_reply(
    messages: list[ModelMessage],
    info: AgentInfo,
) -> ModelResponse:
    returns = _tool_returns(messages)
    if not returns:
        return ModelResponse(
            parts=[ToolCallPart(tool_name="inspect_assay_features_batch", args={})]
        )

    batch = AssayFeatureInspectionBatch.model_validate(returns[-1].content)
    policies = []
    for inspection in batch.inspections:
        excluded = [
            family
            for family in inspection.families
            if family.count > 0 and family.defaultExclude is True
        ]
        protected = [
            family
            for family in inspection.families
            if family.count > 0 and family.defaultExclude is False
        ]
        species = inspection.species
        evidence_ids = list(inspection.evidenceIds)
        if species == "unknown":
            species = "homo_sapiens"
            evidence_ids.append("context:organism")
        policies.append(
            {
                "assay": inspection.assay,
                "species": species,
                "speciesConfidence": (
                    "high" if inspection.species != "unknown" else "medium"
                ),
                "speciesRationale": (
                    inspection.speciesReason
                    or "The inspected features and caller context support this species."
                ),
                "excludeFamilies": [family.family for family in excluded],
                "protectFamilies": [family.family for family in protected],
                "rationale": (
                    "Use observed technical families as feature-selection exclusions "
                    "while preserving protected biological families."
                ),
                "evidenceIds": list(dict.fromkeys(evidence_ids)),
            }
        )
    return _structured_output(info, {"status": "done", "policies": policies})


async def _experimental_reply(
    messages: list[ModelMessage],
    info: AgentInfo,
) -> ModelResponse:
    returns = _tool_returns(messages)
    if not returns:
        return ModelResponse(
            parts=[ToolCallPart(tool_name="inspect_cell_covariates", args={})]
        )
    if len(returns) == 1:
        return ModelResponse(
            parts=[
                ToolCallPart(
                    tool_name="analyze_experimental_design",
                    args={
                        "column_domains": {},
                        "coefficients_of_interest": [],
                        "units_of_inference": {},
                        "batch_columns": [],
                    },
                )
            ]
        )

    design = CovariateEvidence.model_validate(returns[-1].content)
    evidence_id = design.evidenceIds[0]
    return _structured_output(
        info,
        {
            "batchCorrection": {
                "action": "skip",
                "rationale": (
                    "No explicit technical batch or biological contrast was supplied."
                ),
                "evidenceIds": [evidence_id],
            },
            "rationale": "Continue with an uncorrected baseline.",
            "evidenceIds": [evidence_id],
        },
    )


async def _tuning_reply(
    _messages: list[ModelMessage],
    info: AgentInfo,
) -> ModelResponse:
    return _structured_output(
        info,
        {
            "status": "done",
            "recommendedCandidateId": "baseline",
            "confidence": "medium",
            "rationale": "The single authorized baseline completed successfully.",
            "evidenceIds": ["candidate:baseline:clusters"],
            "stopReason": "The authorized candidate was evaluated.",
        },
    )


async def _biology_reply(
    messages: list[ModelMessage],
    info: AgentInfo,
) -> ModelResponse:
    returns = _tool_returns(messages)
    if not returns:
        return ModelResponse(
            parts=[ToolCallPart(tool_name="inspect_cluster_composition", args={})]
        )
    if len(returns) == 1:
        composition = ClusterCompositionEvidence.model_validate(returns[-1].content)
        cluster_id = sorted(
            composition.clusterCounts,
            key=lambda value: (-composition.clusterCounts[value], value),
        )[0]
        return ModelResponse(
            parts=[
                ToolCallPart(
                    tool_name="inspect_cluster_markers_batch",
                    args={"cluster_ids": [cluster_id]},
                )
            ]
        )

    marker_batch = ClusterMarkerBatchEvidence.model_validate(returns[-1].content)
    marker = next(
        (item for item in marker_batch.clusters if item.evidenceId),
        None,
    )
    if marker is None:
        return _structured_output(
            info,
            {
                "status": "needsInput",
                "needsInput": {
                    "question": "No markers passed the bounded search thresholds.",
                    "requiredInputs": ["markerArtifact"],
                },
                "limitations": marker_batch.warnings,
                "stopReason": "Marker evidence was unavailable.",
            },
        )

    names = [item.featureName or item.featureId for item in marker.markers[:3]]
    return _structured_output(
        info,
        {
            "status": "done",
            "clusterInterpretations": [
                {
                    "clusterId": marker.clusterId,
                    "proposedIdentity": "unresolved marker-defined cluster",
                    "identityIsHypothesis": True,
                    "confidence": "low",
                    "rationale": f"Top returned marker features: {', '.join(names)}.",
                    "evidenceIds": [marker.evidenceId],
                }
            ],
            "evidenceIds": [marker.evidenceId],
            "limitations": [
                "The scripted documentation model does not assign cell identities."
            ],
            "stopReason": "One bounded cluster was reviewed.",
        },
    )


enrichment_model = FunctionModel(_enrichment_reply)
experimental_model = FunctionModel(_experimental_reply)
tuning_model = FunctionModel(_tuning_reply)
biology_model = FunctionModel(_biology_reply)
config = AgentRunConfig(temperature=0.0, timeoutSeconds=600.0)

In [3]:
enrichment = DataEnrichmentAgent(enrichment_model, config=config).run(
    ds,
    context=DataEnrichmentContext(
        studyContext=(
            "10x 5K PBMC RNA-seq from peripheral blood of a healthy human donor."
        ),
        organismHint="human",
        tissueReferences=["peripheral blood"],
        cellTypeReferences=["T cell", "B cell", "NK cell", "monocyte"],
        experimentalDetails=["10x 3 prime RNA-seq", "single donor"],
    ),
    assays=["RNA"],
)

policy = enrichment.policies[0]
{
    "status": enrichment.status,
    "species": policy.species,
    "exclude_families": policy.excludeFamilies,
    "protect_families": policy.protectFamilies,
    "tool_calls": [call.name for call in enrichment.toolCalls],
}

{'status': 'done',
 'species': 'homo_sapiens',
 'exclude_families': ['ribosomal', 'histone'],
 'protect_families': ['cellCycle'],
 'tool_calls': ['inspect_assay_features_batch']}

In [4]:
ds.filter_cells(
    attrs=["RNA_nCounts", "RNA_nFeatures", "RNA_percentMito"],
    highs=[15000, 4000, 15],
    lows=[1000, 500, 0],
    reset_previous=True,
)
hvg_ref = ds.mark_hvgs(
    from_assay="RNA",
    top_n=500,
    min_cells=20,
    show_plot=False,
)
normalized = ds.run_normalization(
    from_assay="RNA",
    features=hvg_ref,
    log_transform=True,
    renormalize_subset=True,
    update_state=False,
)

{
    "active_cells": int(ds.cells.fetch_all("I").sum()),
    "feature_selection": hvg_ref.artifact_id,
    "normalized": normalized.artifact_id,
}

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Calculating feature statistics: 30 / 30 complete

Writing data: 1 / 1 complete

{'active_cells': 3940,
 'feature_selection': 'e87d12d8bef0b07af358ec92cb218ee72cafa64aa782575271680be8a09fa46e',
 'normalized': 'ed33b168e658dcff9cacd7bb5f9fc61ca8dd1b80e3e4bb6aeddc95eb9ef1ee24'}

In [5]:
experimental = ExperimentalContextAgent(
    experimental_model,
    config=config,
).run(
    ds,
    study_context=(
        "Healthy-donor 5K PBMC. No treatment or batch labels are available. "
        "Do not invent a technical batch or biological contrast."
    ),
)

{
    "status": experimental.status,
    "batch_action": experimental.decision.batchCorrection.action,
    "batch_columns": experimental.decision.batchCorrection.batchColumns,
    "coefficients": experimental.decision.coefficientsOfInterest,
}

{'status': 'done',
 'batch_action': 'skip',
 'batch_columns': [],
 'coefficients': []}

In [6]:
if experimental.status != "done":
    raise RuntimeError(f"Experimental Context stopped with {experimental.status!r}")

tuning_handoff = experimental.to_parameter_tuning_handoff()
candidate = ParameterCandidate(
    candidateId="baseline",
    dimensions=15,
    leidenResolution=0.5,
    neighborsK=11,
    useHarmony=False,
)
tuning = ParameterTuningAgent(tuning_model, config=config).run(
    ds,
    normalized=normalized,
    from_assay="RNA",
    candidates=[candidate],
    experimental_handoff=tuning_handoff,
    max_candidates=1,
    max_refined_candidates=0,
    min_cluster_cells=10,
)

evaluation = tuning.evaluations[0]
{
    "status": tuning.status,
    "recommended_candidate": tuning.recommendedCandidateId,
    "eligible": evaluation.eligible,
    "clusters": evaluation.metrics.nClusters,
    "smallest_cluster": evaluation.metrics.minClusterCells,
    "cluster_column": evaluation.clusterColumn,
}

Fitting PCA: 1 / 1 complete

Writing reduced coordinates: 1 / 1 complete

Calculating reduced coordinates: 1 / 1 complete

Fitting ANN: 1 / 1 complete

Identifying neighbors: 1 / 1 complete

Calculating silhouette scores: 10 / 10 complete

{'status': 'done',
 'recommended_candidate': 'baseline',
 'eligible': True,
 'clusters': 10,
 'smallest_cluster': 15,
 'cluster_column': 'RNA_agent_tuning_baseline_c99497526b04'}

In [7]:
if tuning.status != "done":
    raise RuntimeError(f"Parameter Tuning stopped with {tuning.status!r}")

biology_handoff = tuning.to_biological_handoff()
biology = BiologicalInterpretationAgent(
    biology_model,
    config=config,
).run(
    ds,
    tuning_handoff=biology_handoff,
    biological_context=BiologicalContext(
        organism="Homo sapiens",
        tissue="peripheral blood",
        cellTypeReferences=["T cell", "B cell", "NK cell", "monocyte"],
        experimentalDetails=["healthy donor PBMC", "no treatment contrast"],
    ),
    allow_marker_search=True,
    marker_features=hvg_ref,
    max_clusters=1,
    max_markers=5,
    marker_min_score=0.01,
    marker_min_fraction=0.0,
)

{
    "status": biology.status,
    "interpretations": [
        {
            "cluster": item.clusterId,
            "identity": item.proposedIdentity,
            "rationale": item.rationale,
            "evidence": item.evidenceIds,
        }
        for item in biology.clusterInterpretations
    ],
    "tool_calls": [call.toolName for call in biology.runInfo.toolCalls],
    "treatment_observations": len(biology.treatmentObservations),
    "limitations": biology.limitations,
}

Finding markers: 1 / 1 complete

{'status': 'done',
 'interpretations': [{'cluster': '4',
   'identity': 'unresolved marker-defined cluster',
   'rationale': 'Top returned marker features: BEX3, TCF7, SOCS3.',
   'evidence': ['markers:5acbeb14799917987878303dc88244d883b8d8bdd361142ec67c3b18af98d223:clusters:d30cd7bca06d1c40afd5a808ef608ec58653b8f3f144f72a8be91c0b7713c21e:cluster:4']}],
 'tool_calls': ['inspect_cluster_composition',
  'inspect_cluster_markers_batch'],
 'treatment_observations': 0,
 'limitations': ['The scripted documentation model does not assign cell identities.']}

In [8]:
cluster_column = biology_handoff.clusterColumn
ds.cells.to_pandas_dataframe(
    columns=[cluster_column],
    key="I",
)[cluster_column].value_counts().sort_index()

RNA_agent_tuning_baseline_c99497526b04
1      712
2       27
3      235
4     1267
5      318
6      507
7      143
8      259
9      457
10      15
Name: count, dtype: int64